In [2]:
import os
import subprocess

LOCAL_DATA_DIR = "data"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

BUCKET = "s3://humanoid-damage-cnn-834257582282-us-east-1-an/processed/damage_detection"

# Sincronizamos ambas carpetas (damaged / undamaged) a disco local
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/damaged/", f"{LOCAL_DATA_DIR}/damaged/"], check=True)
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/undamaged/", f"{LOCAL_DATA_DIR}/undamaged/"], check=True)

# Revisamos cuántas imágenes hay de cada clase
n_damaged = len(os.listdir(f"{LOCAL_DATA_DIR}/damaged"))
n_undamaged = len(os.listdir(f"{LOCAL_DATA_DIR}/undamaged"))
print(f"Dañadas: {n_damaged} | No dañadas: {n_undamaged}")

Dañadas: 11194 | No dañadas: 12981


In [11]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_DATA_DIR,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_DATA_DIR,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_batches = tf.data.experimental.cardinality(val_test_ds)
val_ds = val_test_ds.take(val_batches // 2)
test_ds = val_test_ds.skip(val_batches // 2)

class_names = train_ds.class_names
print("Clases:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

Found 24174 files belonging to 2 classes.
Using 16922 files for training.
Found 24174 files belonging to 2 classes.
Using 7252 files for validation.
Clases: ['damaged', 'undamaged']


In [3]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.15),
])

In [4]:
from tensorflow.keras import layers, models

def build_cnn(input_shape=(224, 224, 3)):
    inputs = tf.keras.Input(shape=input_shape)
    
    x = layers.Rescaling(1./255)(inputs)
    x = data_augmentation(x)
    
    # Bloque 1
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    
    # Bloque 2
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    
    # Bloque 3
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    
    # Bloque 4
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    
    outputs = layers.Dense(1, activation="sigmoid")(x)
    
    return models.Model(inputs, outputs)

model = build_cnn()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 423,361 (1.61 MB)

 Trainable params: 422,401 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

In [4]:
import tensorflow as tf

# Cargar el modelo desde el mejor checkpoint guardado
model = tf.keras.models.load_model("best_model.keras")

print("Modelo cargado correctamente")
model.summary()

Modelo cargado correctamente


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,268,165 (4.84 MB)

 Trainable params: 422,401 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

 Optimizer params: 844,804 (3.22 MB)

In [5]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", monitor="val_recall", mode="max", save_best_only=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50
496/529 ━━━━━━━━━━━━━━━━━━━━ 55s 2s/step - accuracy: 0.8593 - loss: 0.3398 - precision: 0.8657 - recall: 0.8694

Invalid SOS parameters for sequential JPEG


529/529 ━━━━━━━━━━━━━━━━━━━━ 932s 2s/step - accuracy: 0.8802 - loss: 0.2998 - precision: 0.8813 - recall: 0.8967 - val_accuracy: 0.8078 - val_loss: 0.4316 - val_precision: 0.9640 - val_recall: 0.6704 - learning_rate: 0.0010
Epoch 2/50
529/529 ━━━━━━━━━━━━━━━━━━━━ 859s 2s/step - accuracy: 0.9094 - loss: 0.2348 - precision: 0.9100 - recall: 0.9218 - val_accuracy: 0.8977 - val_loss: 0.2619 - val_precision: 0.8565 - val_recall: 0.9745 - learning_rate: 0.0010
Epoch 3/50
529/529 ━━━━━━━━━━━━━━━━━━━━ 859s 2s/step - accuracy: 0.9190 - loss: 0.2105 - precision: 0.9225 - recall: 0.9265 - val_accuracy: 0.8778 - val_loss: 0.3535 - val_precision: 0.8280 - val_recall: 0.9776 - learning_rate: 0.0010
Epoch 4/50
529/529 ━━━━━━━━━━━━━━━━━━━━ 856s 2s/step - accuracy: 0.9236 - loss: 0.1993 - precision: 0.9277 - recall: 0.9297 - val_accuracy: 0.8861 - val_loss: 0.2689 - val_precision: 0.9674 - val_recall: 0.8173 - learning_rate: 0.0010
Epoch 5/50
529/529 ━━━━━━━━━━━━━━━━━━━━ 855s 2s/step - accuracy: 0.9281

In [5]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", monitor="val_recall", mode="max", save_best_only=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,  # menos épocas ya que retomamos, no arrancamos de cero
    callbacks=callbacks
)

Epoch 1/30
1009/1058 ━━━━━━━━━━━━━━━━━━━━ 38s 785ms/step - accuracy: 0.9536 - loss: 0.1237 - precision: 0.9607 - recall: 0.9522

Invalid SOS parameters for sequential JPEG


1058/1058 ━━━━━━━━━━━━━━━━━━━━ 872s 821ms/step - accuracy: 0.9532 - loss: 0.1235 - precision: 0.9612 - recall: 0.9508 - val_accuracy: 0.9345 - val_loss: 0.1572 - val_precision: 0.9491 - val_recall: 0.9288 - learning_rate: 2.5000e-04
Epoch 2/30
1005/1058 ━━━━━━━━━━━━━━━━━━━━ 41s 791ms/step - accuracy: 0.9523 - loss: 0.1200 - precision: 0.9602 - recall: 0.9499

Invalid SOS parameters for sequential JPEG


1058/1058 ━━━━━━━━━━━━━━━━━━━━ 876s 827ms/step - accuracy: 0.9552 - loss: 0.1175 - precision: 0.9625 - recall: 0.9534 - val_accuracy: 0.9185 - val_loss: 0.2114 - val_precision: 0.9026 - val_recall: 0.9515 - learning_rate: 2.5000e-04
Epoch 3/30
1002/1058 ━━━━━━━━━━━━━━━━━━━━ 44s 793ms/step - accuracy: 0.9603 - loss: 0.1056 - precision: 0.9655 - recall: 0.9600

Invalid SOS parameters for sequential JPEG


... (13 outputs left out)

Invalid SOS parameters for sequential JPEG


1058/1058 ━━━━━━━━━━━━━━━━━━━━ 876s 828ms/step - accuracy: 0.9630 - loss: 0.0970 - precision: 0.9690 - recall: 0.9616 - val_accuracy: 0.7740 - val_loss: 0.8718 - val_precision: 0.7193 - val_recall: 0.9531 - learning_rate: 1.2500e-04
Epoch 11/30
1009/1058 ━━━━━━━━━━━━━━━━━━━━ 38s 795ms/step - accuracy: 0.9634 - loss: 0.0947 - precision: 0.9708 - recall: 0.9605

Invalid SOS parameters for sequential JPEG


1058/1058 ━━━━━━━━━━━━━━━━━━━━ 880s 831ms/step - accuracy: 0.9627 - loss: 0.0978 - precision: 0.9691 - recall: 0.9609 - val_accuracy: 0.9353 - val_loss: 0.1681 - val_precision: 0.9185 - val_recall: 0.9658 - learning_rate: 1.2500e-04


In [4]:
import gc
from tensorflow.keras.callbacks import Callback

class MemoryCleanup(Callback):
    def on_epoch_end(self, epoch, logs=None):
        gc.collect()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", monitor="val_recall", mode="max", save_best_only=True),
    MemoryCleanup()
]

In [5]:
model = tf.keras.models.load_model("best_model.keras")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

Epoch 1/5
1009/1058 ━━━━━━━━━━━━━━━━━━━━ 38s 794ms/step - accuracy: 0.9600 - loss: 0.1041 - precision: 0.9666 - recall: 0.9583

Invalid SOS parameters for sequential JPEG


1058/1058 ━━━━━━━━━━━━━━━━━━━━ 882s 830ms/step - accuracy: 0.9583 - loss: 0.1082 - precision: 0.9653 - recall: 0.9565 - val_accuracy: 0.8422 - val_loss: 0.3404 - val_precision: 0.8429 - val_recall: 0.8708 - learning_rate: 2.5000e-04
Epoch 2/5
1004/1058 ━━━━━━━━━━━━━━━━━━━━ 43s 806ms/step - accuracy: 0.9619 - loss: 0.1030 - precision: 0.9689 - recall: 0.9594

Invalid SOS parameters for sequential JPEG


1058/1058 ━━━━━━━━━━━━━━━━━━━━ 892s 842ms/step - accuracy: 0.9612 - loss: 0.1043 - precision: 0.9687 - recall: 0.9585 - val_accuracy: 0.9444 - val_loss: 0.1516 - val_precision: 0.9266 - val_recall: 0.9740 - learning_rate: 2.5000e-04
Epoch 3/5
1001/1058 ━━━━━━━━━━━━━━━━━━━━ 45s 798ms/step - accuracy: 0.9603 - loss: 0.1031 - precision: 0.9680 - recall: 0.9574

Invalid SOS parameters for sequential JPEG


1058/1058 ━━━━━━━━━━━━━━━━━━━━ 882s 833ms/step - accuracy: 0.9598 - loss: 0.1060 - precision: 0.9672 - recall: 0.9572 - val_accuracy: 0.9452 - val_loss: 0.1410 - val_precision: 0.9505 - val_recall: 0.9481 - learning_rate: 2.5000e-04
Epoch 4/5
1002/1058 ━━━━━━━━━━━━━━━━━━━━ 44s 797ms/step - accuracy: 0.9627 - loss: 0.0959 - precision: 0.9698 - recall: 0.9600

Invalid SOS parameters for sequential JPEG


1058/1058 ━━━━━━━━━━━━━━━━━━━━ 882s 833ms/step - accuracy: 0.9601 - loss: 0.1025 - precision: 0.9672 - recall: 0.9579 - val_accuracy: 0.9339 - val_loss: 0.2050 - val_precision: 0.9143 - val_recall: 0.9684 - learning_rate: 2.5000e-04
Epoch 5/5
1007/1058 ━━━━━━━━━━━━━━━━━━━━ 40s 793ms/step - accuracy: 0.9627 - loss: 0.1002 - precision: 0.9726 - recall: 0.9570

Invalid SOS parameters for sequential JPEG


1058/1058 ━━━━━━━━━━━━━━━━━━━━ 878s 830ms/step - accuracy: 0.9622 - loss: 0.1026 - precision: 0.9704 - recall: 0.9586 - val_accuracy: 0.9331 - val_loss: 0.1807 - val_precision: 0.9129 - val_recall: 0.9684 - learning_rate: 2.5000e-04


In [9]:
model = tf.keras.models.load_model("best_model.keras")

In [12]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_true = []
y_pred_prob = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.append(labels.numpy())
    y_pred_prob.append(preds)

y_true = np.concatenate(y_true)
y_pred_prob = np.concatenate(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=class_names))
print("Matriz de confusión:\n", confusion_matrix(y_true, y_pred))
print("ROC-AUC:", roc_auc_score(y_true, y_pred_prob))

Invalid SOS parameters for sequential JPEG


              precision    recall  f1-score   support

     damaged       0.97      0.90      0.94      1657
   undamaged       0.92      0.98      0.95      1963

    accuracy                           0.94      3620
   macro avg       0.95      0.94      0.94      3620
weighted avg       0.95      0.94      0.94      3620

Matriz de confusión:
 [[1497  160]
 [  43 1920]]
ROC-AUC: 0.9906715700938085


2026-08-01 02:06:40.543848: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
